# 5.4 Markov Random Fields

Undirected models express mutual compatibility, then one global partition function turns scores into probabilities.

MRFs replace directed conditionals with clique potentials. They lead naturally to factor graphs, belief propagation, and conditional random fields.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5404)


def enumerate_mrf(n_vars, unary, edges, beta):
    assignments = list(itertools.product([0, 1], repeat=n_vars))
    scores = []
    for assignment in assignments:
        score = 1.0
        for i, value in enumerate(assignment):
            score = score * unary[i][value]
        for i, j in edges:
            same = assignment[i] == assignment[j]
            score = score * math.exp(beta if same else -beta)
        scores.append(score)
    scores = np.asarray(scores, dtype=float)
    probabilities = scores / scores.sum()
    marginals = np.zeros((n_vars, 2), dtype=float)
    for assignment, probability in zip(assignments, probabilities):
        for i, value in enumerate(assignment):
            marginals[i, value] = marginals[i, value] + probability
    return assignments, probabilities, marginals, float(scores.sum())


def mean_marginal_error(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))


def mean_field_mrf(n_vars, unary, edges, beta, iterations=40):
    q = np.full((n_vars, 2), 0.5, dtype=float)
    neighbors = {i: [] for i in range(n_vars)}
    for i, j in edges:
        neighbors[i].append(j)
        neighbors[j].append(i)
    for step in range(iterations):
        new_q = q.copy()
        for i in range(n_vars):
            logits = np.log(unary[i])
            for j in neighbors[i]:
                logits[0] = logits[0] + beta * q[j, 0] - beta * q[j, 1]
                logits[1] = logits[1] - beta * q[j, 0] + beta * q[j, 1]
            logits = logits - logits.max()
            probs = np.exp(logits)
            new_q[i] = probs / probs.sum()
        q = new_q
    return q


## The concept, built once (D1)

An MRF multiplies nonnegative clique potentials and then divides by one global partition function.

$$p(x)=\frac{1}{Z}\prod_{c\in\mathcal C}\psi_c(x_c),\quad Z=\sum_x\prod_{c\in\mathcal C}\psi_c(x_c)$$

The reusable enumerator computes scores, $Z$, probabilities, and node marginals for binary pairwise MRFs.

In [ ]:

def d1_pair_mrf():
    unary = [np.array([1.0, 1.0]), np.array([1.0, 1.0])]
    edges = [(0, 1)]
    assignments, probabilities, marginals, partition = enumerate_mrf(2, unary, edges, 0.8)
    return assignments, probabilities, marginals, partition


For the lesson pairwise model, aligned score $e^{0.8}=2.225541$, opposed score $e^{-0.8}=0.449329$, $Z=5.349740$, one aligned-state probability $0.416009$, and marginal $0.500$.

In [ ]:

assignments, probabilities, marginals, partition = d1_pair_mrf()
aligned = math.exp(0.8)
opposed = math.exp(-0.8)
assert abs(aligned - 2.225540928492468) < 1e-12
assert abs(opposed - 0.44932896411722156) < 1e-12
assert abs(partition - 5.349739785219379) < 1e-12
assert abs(probabilities[0] - 0.4160090156869525) < 1e-12
assert abs(marginals[0, 1] - 0.5) < 1e-12
print("partition", partition)
print("marginals", marginals)


## The dataset ladder

The ladder moves from a two-node exact model to a four-node chain, a bimodal square, a correlated two-dimensional denoising grid, and a larger strong-coupling grid approximated with mean field against exact reference.

In [ ]:

rungs = []

unary = [np.ones(2), np.ones(2)]
edges = [(0, 1)]
assignments, probabilities, truth, partition = enumerate_mrf(2, unary, edges, 0.8)
rungs.append({"name": "D1 pair", "truth": truth, "estimate": truth, "shape": (1, 2)})

unary = [np.array([1.1, 0.9]), np.ones(2), np.ones(2), np.array([0.9, 1.1])]
edges = [(0, 1), (1, 2), (2, 3)]
assignments, probabilities, truth, partition = enumerate_mrf(4, unary, edges, 0.7)
estimate = mean_field_mrf(4, unary, edges, 0.7)
rungs.append({"name": "D2 chain", "truth": truth, "estimate": estimate, "shape": (1, 4)})

unary = [np.ones(2) for _ in range(4)]
edges = [(0, 1), (1, 3), (3, 2), (2, 0)]
assignments, probabilities, truth, partition = enumerate_mrf(4, unary, edges, 1.2)
estimate = mean_field_mrf(4, unary, edges, 1.2)
rungs.append({"name": "D3 bimodal square", "truth": truth, "estimate": estimate, "shape": (2, 2)})

observed = np.array([[0, 0, 1], [0, 1, 1], [1, 1, 1]])
unary = []
for value in observed.ravel():
    unary.append(np.array([2.0, 0.5]) if value == 0 else np.array([0.5, 2.0]))
edges = []
for r in range(3):
    for c in range(3):
        idx = r * 3 + c
        if r < 2:
            edges.append((idx, idx + 3))
        if c < 2:
            edges.append((idx, idx + 1))
assignments, probabilities, truth, partition = enumerate_mrf(9, unary, edges, 0.6)
estimate = mean_field_mrf(9, unary, edges, 0.6)
rungs.append({"name": "D4 denoising grid", "truth": truth, "estimate": estimate, "shape": (3, 3)})

observed = rng.integers(0, 2, size=(4, 4))
unary = []
for value in observed.ravel():
    unary.append(np.array([2.5, 0.4]) if value == 0 else np.array([0.4, 2.5]))
edges = []
for r in range(4):
    for c in range(4):
        idx = r * 4 + c
        if r < 3:
            edges.append((idx, idx + 4))
        if c < 3:
            edges.append((idx, idx + 1))
assignments, probabilities, truth, partition = enumerate_mrf(16, unary, edges, 0.9)
estimate = mean_field_mrf(16, unary, edges, 0.9)
rungs.append({"name": "D5 strong grid", "truth": truth, "estimate": estimate, "shape": (4, 4), "partition": partition, "max_score": float(np.max(probabilities) * partition)})

for rung in rungs:
    print(rung["name"], "nodes", rung["truth"].shape[0], "sample p1", np.round(rung["truth"][:4, 1], 3))


In [ ]:

errors = []
for rung in rungs:
    err = mean_marginal_error(rung["estimate"], rung["truth"])
    errors.append(err)
    print(f"{rung['name']}: marginal_error={err:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(rungs):
    exact = rung["truth"][:, 1]
    estimated = rung["estimate"][:, 1]
    x = np.arange(exact.shape[0])
    axes[0, i].bar(x - 0.2, exact, width=0.4, label="exact")
    axes[0, i].bar(x + 0.2, estimated, width=0.4, label="estimated")
    axes[0, i].set_ylim(0, 1)
    axes[0, i].set_title(rung["name"])
    axes[1, i].axis("off")
axes[0, 0].legend()
axes[1, 2].plot(range(1, 6), errors, marker="o")
axes[1, 2].set_xticks(range(1, 6))
axes[1, 2].set_xlabel("rung")
axes[1, 2].set_ylabel("marginal error")
axes[1, 2].set_title("error vs iteration target")
fig.tight_layout()


## Pitfall on D5: ignoring $Z$

Raw potentials can be much larger than one and cannot be read as probabilities. Exact enumeration computes $Z$ first; the normalized marginals are valid probabilities.

In [ ]:

d5 = rungs[-1]
raw_score = d5["max_score"]
partition = d5["partition"]
fixed_probability = raw_score / partition
print("largest D5 raw score", raw_score)
print("D5 partition", partition)
print("fixed D5 assignment probability", fixed_probability)
assert raw_score > 1.0
assert 0.0 <= fixed_probability <= 1.0


## Evaluate it + Practice

- Metric: mean absolute node-marginal error against exact enumeration or a high-precision reference.
- No-skill baseline: use the prior, unary factors, or a single local conditional without global inference.
- Cheap sanity check: symmetric zero-field D1 has both node marginals exactly 0.500.
- Ablation: set beta to zero and watch compatibility-driven smoothing disappear.
- Failure signals: partition overflow, raw potentials treated as probabilities, or mean-field locking onto one mode.

Practice prompts:
1. Increase D3 beta and inspect bimodality.

2. Flip one D4 observed pixel and rerun inference.

3. Compare mean-field initialization choices on D5.